# Block-level storage warming analysis

Consumes the per-block savings parquet emitted by
`access_savings_parser.py --per-block-out per_block_savings.parquet`.

Pipeline:
1. `python3 extract_block.py --start N --end M --out DIR` (writes `block_*.parquet` and `meta_*.parquet`)
2. `python3 access_savings_parser.py --dir DIR --windows 0,1,...,31 --per-block-out DIR/per_block_savings.parquet`
3. Run this notebook against the per-block parquet.

In [ ]:
import os, glob, re
import numpy as np
import pandas as pd
import plotly.graph_objects as go

DATA_DIR = os.environ.get('BALWARMING_DIR', '/tmp/balwarming')
PER_BLOCK = os.path.join(DATA_DIR, 'per_block_savings.parquet')

color_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
                 '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']

def hex_to_rgba(hex_color, alpha):
    h = hex_color.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f'rgba({r}, {g}, {b}, {alpha})'

In [ ]:
df = pd.read_parquet(PER_BLOCK).sort_values('block_number').reset_index(drop=True)
df['date'] = pd.to_datetime(df['timestamp'], unit='s')
windows_present = sorted(int(c[len('saving_W'):]) for c in df.columns if c.startswith('saving_W'))
print(f'{len(df)} blocks, range {df.block_number.min()}..{df.block_number.max()}, '
      f'{df.date.min()} -> {df.date.max()}; windows: {windows_present}')
df.head()

## Most-accessed (address, slot) pairs
Top-50 keys by access count across the loaded range. Read directly from the row-level parquets.

In [ ]:
block_files = sorted(glob.glob(os.path.join(DATA_DIR, 'block_*.parquet')),
                     key=lambda p: int(re.search(r'block_(\d+)', p).group(1)))
rows = pd.concat([pd.read_parquet(f) for f in block_files], ignore_index=True)
n_blocks = rows.block_number.nunique()
n_txs = rows.groupby('block_number')['tx_index'].nunique().sum()
rows['key'] = rows['address'].fillna('') + '|' + rows['storage_key'].fillna('')
pop = (rows.groupby('key')
           .agg(accesses_per_block=('block_number', lambda s: s.nunique() / n_blocks),
                accesses_per_tx=('key', 'count'))
           .assign(accesses_per_tx=lambda d: d['accesses_per_tx'] / n_txs)
           .sort_values('accesses_per_tx', ascending=False)
           .head(50)
           .reset_index())
pop[['address', 'storage_key']] = pop['key'].str.split('|', n=1, expand=True)
pop['Abbreviated Address'] = pop['address'].apply(lambda x: x[:6] + '...' + x[-4:] if x else '')
pop['Storage Key'] = pop['storage_key'].apply(lambda x: (x[:6] + '...' + x[-4:]) if x and len(x) > 12 else (x or ''))
pop = pop.rename(columns={'accesses_per_block': 'Accesses per Block', 'accesses_per_tx': 'Accesses per Transaction'})
pop[['Abbreviated Address', 'Storage Key', 'Accesses per Block', 'Accesses per Transaction']].round(4).head(20)

## Gas-used time series with hypothetical warming
Each line shows what block-level gas usage would look like under a given warming-window size.

In [ ]:
# Bin to hour if span is large; otherwise show per-block.
span_hours = (df.date.max() - df.date.min()).total_seconds() / 3600
if span_hours >= 4:
    df_t = df.copy()
    df_t['date'] = df_t['date'].dt.ceil('h')
    agg = df_t.groupby('date').agg(
        gas_used=('gas_used', 'mean'),
        block_count=('block_number', 'count'),
        **{f'saving_W{w}': (f'saving_W{w}', 'mean') for w in windows_present},
        max_saving=('max_saving', 'mean'),
    ).reset_index()
else:
    agg = df.rename(columns={'block_number': 'date'}).copy()

fig = go.Figure()
fig.add_trace(go.Scatter(x=agg['date'], y=agg['gas_used'], mode='lines',
                          name='Gas used (currently)', line=dict(width=2, color=color_palette[0])))
for i, w in enumerate([0, 5, 10, 15]):
    if w in windows_present:
        fig.add_trace(go.Scatter(x=agg['date'], y=agg['gas_used'] - agg[f'saving_W{w}'],
                                  mode='lines', name=f'Window={w}',
                                  line=dict(width=2, color=color_palette[i+1])))
fig.update_layout(title='Gas used vs hypothetical warming (multi-block windows)',
                  xaxis_title=('Date' if span_hours >= 4 else 'Block number'),
                  yaxis_title='Gas',
                  template='plotly_white', width=900, height=480, hovermode='x unified',
                  legend=dict(orientation='h', y=-0.2))
fig.show()

## Savings as % of block gas-used

In [ ]:
fig = go.Figure()
if 0 in windows_present:
    fig.add_trace(go.Scatter(x=df['date'], y=-df['pct_saving_W0'], mode='lines',
                              name='Block-level (intra-block, W=0)',
                              fill='tozeroy', line=dict(width=0),
                              fillcolor=hex_to_rgba(color_palette[0], 0.7)))
if 15 in windows_present:
    fig.add_trace(go.Scatter(x=df['date'], y=-df['pct_saving_W15'], mode='lines',
                              name='Multi-block (W=15)',
                              fill='tonexty', line=dict(width=0),
                              fillcolor=hex_to_rgba(color_palette[1], 0.7)))
fig.add_shape(type='line', x0=0, x1=1, y0=0, y1=0, xref='paper',
              line=dict(color='rgba(128,128,128,0.7)', dash='dash'))
fig.update_layout(title='Savings as % of block gas-used (negative = saved)',
                  xaxis_title='Date', yaxis_title='%',
                  template='plotly_white', width=900, height=480, hovermode='x unified',
                  legend=dict(orientation='h', y=-0.2))
fig.show()

## Saving potential vs window size (the headline plot)
Median and mean of per-block percentage savings, across all loaded blocks, as a function of the warming-window size W.

In [ ]:
ws = sorted(w for w in windows_present if w >= 1)
med = [df[f'pct_saving_W{w}'].median() for w in ws]
avg = [df[f'pct_saving_W{w}'].mean() for w in ws]
max_med = df['pct_max_saving'].median()

fig = go.Figure()
fig.add_trace(go.Scatter(x=ws, y=med, mode='lines+markers', name='Median', line=dict(color=color_palette[0])))
fig.add_trace(go.Scatter(x=ws, y=avg, mode='lines+markers', name='Mean', line=dict(color=color_palette[1])))
fig.add_shape(type='line', x0=ws[0], x1=ws[-1], y0=max_med, y1=max_med,
              line=dict(color='rgba(128,128,128,0.7)', dash='dash'))
fig.add_annotation(x=ws[len(ws)//2], y=max_med, text=f'Max if everything warm (median = {max_med:.1f}%)',
                    showarrow=False, yshift=12, bgcolor='rgba(255,255,255,0.8)')
fig.update_layout(title='Multi-block warming potential',
                  xaxis_title='Window size (blocks)',
                  yaxis_title='% savings of block gas used',
                  template='plotly_white', width=900, height=520,
                  legend=dict(x=0.98, y=0.02, xanchor='right', yanchor='bottom'))
fig.show()
pd.DataFrame({'window': ws, 'median_pct': med, 'mean_pct': avg}).round(3)

## Per-opcode breakdown (largest window)

In [ ]:
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath('access_savings_parser.py')))
from access_savings_parser import compute_block_gaps, per_opcode_breakdown, SAVING
rows_g = compute_block_gaps(rows)
Wmax = max(windows_present)
per_opcode_breakdown(rows_g, Wmax).round(4)